# 00 — Push the evaluation dataset to all 3 registries

Like prompts, every platform has a **Datasets** feature: a registry of input/expected-output pairs you can run an agent against and grade. We push the same 4 SCENARIOS (defined in `shared/workflow.py`) as a dataset named `permission-agent-scenarios` to each platform — so Datasets aren't an empty menu item when you screenshot the UIs.

Run this **once when setting up from scratch** (or any time the dataset changes). It's idempotent — re-running is safe.

## What we push

| Item | Input (the prompt) | Expected output | Why |
| --- | --- | --- | --- |
| `one_tool` | "What's the ads_id for Jane Doe?" | `"ADS-1001"` | exact match deterministic |
| `two_tools` | "What permissions does Alice Nguyen currently have?" | list of 3 perms | set comparison |
| `three_tools` | "Does Jane Doe have admin access to billing-prod?" | `True` | boolean check |
| `judgment_call` | "Briefly explain who Jane Doe is..." | `None` (judged by rubric) | LLM-as-judge fit |

## What you'll see at the end

| Platform | UI location after push |
| --- | --- |
| 🔵 Langfuse | `LANGFUSE_HOST` → your project → **Datasets** → `permission-agent-scenarios` |
| 🟢 LangSmith | <https://smith.langchain.com> → **Datasets & Experiments** (workspace-level) → `permission-agent-scenarios` |
| 🟠 Galileo | <https://app.galileo.ai> → project `$GALILEO_PROJECT` → **Datasets** → `permission-agent-scenarios` |

## 1. Load env + SCENARIOS

In [ ]:
import os, sys, pathlib, json
from dotenv import load_dotenv

ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

from shared.workflow import SCENARIOS, DATASET_NAME

print(f"DATASET_NAME : {DATASET_NAME}")
print(f"item count   : {len(SCENARIOS)}")
for sc in SCENARIOS:
    print(f"  - {sc['id']:<14} expected_answer={json.dumps(sc.get('expected_answer'))}")

## 2. 🔵 Langfuse — `create_dataset` + `create_dataset_item`

Langfuse models a dataset as **a named container with items**. You create the dataset once, then push items into it. Items have `input`, `expected_output`, and `metadata` — all free-form JSON.

Idempotency: `create_dataset` is upsert-style; re-running won't error. `create_dataset_item` always creates a new item, so re-running this cell duplicates items. For a demo it's fine; in production you'd `update_dataset_item` instead.

In [ ]:
from langfuse import Langfuse

lf = Langfuse()
assert lf.auth_check(), "Langfuse auth failed"

# Upsert the dataset
lf.create_dataset(
    name=DATASET_NAME,
    description="4 scenarios for the permission-checking agent: 3 deterministic + 1 LLM-judge-fit",
    metadata={"source": "observability_comparison demo"},
)
print(f"  Langfuse: dataset `{DATASET_NAME}` ready")

# Check what's already in it so we don't duplicate items on re-runs
existing = lf.get_dataset(DATASET_NAME)
existing_scenario_ids = {(it.metadata or {}).get("scenario_id") for it in existing.items}

for sc in SCENARIOS:
    if sc["id"] in existing_scenario_ids:
        print(f"     skip (already present): {sc['id']}")
        continue
    lf.create_dataset_item(
        dataset_name=DATASET_NAME,
        input={"question": sc["prompt"]},
        expected_output={"answer": sc.get("expected_answer")},
        metadata={
            "scenario_id": sc["id"],
            "expected_tool_calls": sc["expected_tool_calls"],
            "llm_judge_fit": sc.get("llm_judge_fit", False),
            "rubric": sc.get("rubric"),
        },
    )
    print(f"     pushed item: {sc['id']}")

print(f"  → Open {os.environ.get('LANGFUSE_HOST', 'http://localhost:3000')} → Datasets → {DATASET_NAME}")

## 3. 🟢 LangSmith — `create_dataset` + `create_examples`

LangSmith models a dataset as **Examples with `inputs` and `outputs`** (both dicts). It's workspace-global (not project-scoped). Inputs/outputs schemas are flexible but enforced for consistency once set.

In [ ]:
from langsmith import Client

client = Client()

try:
    dataset = client.create_dataset(
        DATASET_NAME,
        description="4 scenarios for the permission-checking agent (observability_comparison demo)",
    )
    print(f"  LangSmith: created dataset `{DATASET_NAME}` id={dataset.id}")
    needs_examples = True
except Exception as e:
    if "already exists" in str(e).lower() or "conflict" in str(e).lower():
        dataset = client.read_dataset(dataset_name=DATASET_NAME)
        print(f"  LangSmith: dataset `{DATASET_NAME}` already exists, id={dataset.id}")
        # Check if examples are already there
        existing_count = sum(1 for _ in client.list_examples(dataset_id=dataset.id))
        needs_examples = existing_count == 0
        if not needs_examples:
            print(f"     already has {existing_count} examples — skipping push")
    else:
        raise

if needs_examples:
    client.create_examples(
        inputs=[{"question": sc["prompt"]} for sc in SCENARIOS],
        outputs=[{"answer": sc.get("expected_answer")} for sc in SCENARIOS],
        metadata=[
            {
                "scenario_id": sc["id"],
                "expected_tool_calls": sc["expected_tool_calls"],
                "llm_judge_fit": sc.get("llm_judge_fit", False),
                "rubric": sc.get("rubric"),
            }
            for sc in SCENARIOS
        ],
        dataset_id=dataset.id,
    )
    print(f"     pushed {len(SCENARIOS)} examples")

print(f"  → Open https://smith.langchain.com → Datasets & Experiments → {DATASET_NAME}")

## 4. 🟠 Galileo — `create_dataset(content=...)`

Galileo's `create_dataset` takes the **full content in one shot** (list of dicts, dict of lists, or a CSV path) — there's no separate "add row" call. Datasets are **project-scoped**, like Galileo's templates.

**Schema matters**: Galileo expects specific column names — `input`, `generated_output`, `ground_truth`, and `metadata`. Each row dict you pass becomes one row; the keys map directly to columns. If you use other column names (like `expected_output` or `scenario_id`), they don't render in the UI — Galileo packs them into a JSON blob in the `input` column. We learned this the hard way.

- `input` = the user's prompt (plain string)
- `generated_output` = the model's actual output (left empty; filled during Experiments)
- `ground_truth` = the expected answer (JSON-encoded if structured, e.g. a list)
- `metadata` = anything else, JSON-encoded into one column

In [ ]:
from galileo.datasets import create_dataset, get_dataset

project = os.environ.get("GALILEO_PROJECT", "observability-comparison")

# Galileo schema: columns are `input`, `generated_output`, `ground_truth`, `metadata`.
# Anything outside those names gets buried in the input column as JSON.
content = [
    {
        "input": sc["prompt"],
        "ground_truth": (
            json.dumps(sc["expected_answer"])
            if sc.get("expected_answer") is not None
            else ""
        ),
        "metadata": json.dumps({
            "scenario_id": sc["id"],
            "expected_tool_calls": sc["expected_tool_calls"],
            "llm_judge_fit": sc.get("llm_judge_fit", False),
            "rubric": sc.get("rubric"),
        }),
        # `generated_output` intentionally not set — it's filled by Experiments
    }
    for sc in SCENARIOS
]

existing = get_dataset(name=DATASET_NAME, project_name=project)
if existing:
    print(f"  Galileo: dataset `{DATASET_NAME}` already exists, id={existing.id}")
    print("     (skipping push — delete it in the UI or via SDK to re-create with this schema)")
else:
    ds = create_dataset(name=DATASET_NAME, content=content, project_name=project)
    print(f"  Galileo: pushed dataset `{DATASET_NAME}` to `{project}`, id={ds.id}")
    print(f"     columns: input, ground_truth, metadata (generated_output filled by Experiments)")

print(f"  → Open https://app.galileo.ai → {project} → Datasets → {DATASET_NAME}")

## 5. Verify all three by listing items back

In [ ]:
print("\n--- Langfuse ---")
ds = lf.get_dataset(DATASET_NAME)
print(f"  {DATASET_NAME}: {len(ds.items)} items")
for it in ds.items:
    sid = (it.metadata or {}).get("scenario_id", "?")
    print(f"    - {sid:<14} input={str(it.input)[:80]}")

print("\n--- LangSmith ---")
examples = list(client.list_examples(dataset_id=dataset.id))
print(f"  {DATASET_NAME}: {len(examples)} examples")
for ex in examples:
    sid = (ex.metadata or {}).get("scenario_id", "?")
    print(f"    - {sid:<14} inputs={str(ex.inputs)[:80]}")

print("\n--- Galileo ---")
g = get_dataset(name=DATASET_NAME, project_name=project)
print(f"  {DATASET_NAME}: id={g.id}")
# Galileo's dataset content is fetched separately by version; we just confirm it exists.
print(f"  (count visible in UI: should be {len(SCENARIOS)} rows)")

## What's different between the three APIs (cheat-sheet)

| | Langfuse | LangSmith | Galileo |
| --- | --- | --- | --- |
| **Create + populate** | Two steps: `create_dataset`, then `create_dataset_item` per row | Two steps: `create_dataset`, then `create_examples` (batch) | One step: `create_dataset(content=[...])` |
| **Schema** | Free-form input/expected_output JSON | `inputs`/`outputs` dicts; soft-validated schema | Flat dict per row (no input/output distinction) |
| **Scoping** | Project-global | Workspace-global | **Project-scoped** |
| **Idempotency on dataset name** | Upsert (no error) | 409 on duplicate, requires read-back | Returns existing via `get_dataset` |
| **Item-level versioning** | No (items are mutable) | Yes (examples have versions) | Yes (dataset has versions) |

## Next step — running experiments

We're **not** running experiments in this notebook (intentional — the dataset feature lives on its own). When you want to run them later, each platform offers an `evaluate()` / `run_experiment()` API that takes the dataset + your agent + optional evaluators and produces a comparable run.

If you want to see those in action, that's the next notebook we'd add (likely `04_experiments.ipynb`).